# 2 · Trend

**Trend** is the slowest-moving part of a series: a persistent change in its mean.

**Process:**
1. **See it** with a moving-average plot. Use a window longer than any season (365 for daily data) so the seasons average out.
2. **Model it** with polynomial features of the time dummy (`DeterministicProcess`).
3. **Forecast it** by generating the same features for future dates (`out_of_sample`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 4))
plot_params = dict(color="0.75", style=".-", markeredgecolor="0.25", markerfacecolor="0.25", legend=False)

DATA = Path("../input/store-sales-time-series-forecasting")   # change to where you unzipped the data

In [ ]:
store_sales = pd.read_csv(
    DATA / "train.csv",
    usecols=["store_nbr", "family", "date", "sales", "onpromotion"],
    dtype={"store_nbr": "category", "family": "category", "sales": "float32", "onpromotion": "uint32"},
    parse_dates=["date"],
)
store_sales["date"] = store_sales.date.dt.to_period("D")
store_sales = store_sales.set_index(["store_nbr", "family", "date"]).sort_index()

average_sales = store_sales.groupby("date").mean()["sales"]

## 1. Moving-average plot

In [ ]:
moving_average = average_sales.rolling(
    window=365,       # longer than the longest season
    center=True,      # average sits at the middle of the window
    min_periods=183,  # about half the window
).mean()

ax = average_sales.plot(style=".", color="0.5")
moving_average.plot(ax=ax, linewidth=3, title="365-day moving average", legend=False);

## 2. Trend features with `DeterministicProcess`
`order` is the polynomial degree: 1 for linear, 2 for quadratic, and so on. `constant=True` adds the intercept column, so set `fit_intercept=False`.

In [ ]:
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import DeterministicProcess

y = average_sales.copy()

dp = DeterministicProcess(
    index=y.index,   # dates from the training data
    constant=True,   # dummy feature for the bias
    order=3,         # cubic trend
    drop=True,       # drop terms if needed to avoid collinearity
)
X = dp.in_sample()
X.head()

## 3. Fit and forecast

In [ ]:
model = LinearRegression(fit_intercept=False).fit(X, y)
y_pred = pd.Series(model.predict(X), index=X.index)

X_fore = dp.out_of_sample(steps=90)               # features for the next 90 days
y_fore = pd.Series(model.predict(X_fore), index=X_fore.index)

ax = y.plot(**plot_params, alpha=0.5, title="Trend + 90-day forecast")
y_pred.plot(ax=ax, linewidth=3, label="Trend")
y_fore.plot(ax=ax, linewidth=3, label="Forecast", color="C3")
ax.legend();

## ⚠️ High-order polynomials
They fit the history well and forecast badly. Past the end of the data the highest power takes over and the curve runs off.

In [ ]:
dp = DeterministicProcess(index=y.index, order=11)
model = LinearRegression().fit(dp.in_sample(), y)

X_fore = dp.out_of_sample(steps=90)
ax = y.plot(**plot_params, alpha=0.5, title="Order-11 trend")
pd.Series(model.predict(dp.in_sample()), index=y.index).plot(ax=ax, linewidth=3)
pd.Series(model.predict(X_fore), index=X_fore.index).plot(ax=ax, linewidth=3, color="C3");

**Takeaways**
- Choose the trend shape from the moving-average plot, and keep the order low (1–2).
- Trend models are a good baseline, and they become model 1 of a hybrid (lesson 5), because trees can't extrapolate a trend.